# Homework 2026: the DCDC specimen

*Fracture mechanics, M2. Starting notebook.*

The specimen is a prism with a hole of radius $R$ drilled through its centre,
compressed along its long axis by a uniform stress $\sigma$. Two cracks of
length $a$ leave the hole along that axis. They open in **mode I**, although
the specimen carries no tensile load: the hoop stress at the two points of the
hole that lie on the axis of the load is a tension, and it is what drives them.

The two mid-planes are planes of symmetry, so we model one **quarter**: the
compression along $y$, the width along $x$, the hole centred at the origin.

This notebook does four things:

1. it builds the mesh;
2. it solves the elastic problem and returns the potential energy;
3. it plots the opening and the stress along the crack line;
4. it computes $G$ with the $G$--$\theta$ method.

**What is left for you to write** is marked `TO FILL IN`. Each blank is one
modelling decision, not one line of syntax: a few values, or the body of a
function whose signature and docstring are given.

1. [the two conditions of symmetry](#fill-bcs) — the component each one blocks,
   and its value;
2. [Hooke's law](#fill-hooke) — the function `sigma(e)`, the stress from the
   strain;
3. [the traction on the end face](#fill-loading) — its two components;
4. [the potential energy](#fill-energy) — the function `potential_energy(uh)`;
5. [the crack opening](#fill-opening) — the function `crack_opening(uh, points)`;
6. [the $G$--$\theta$ integral](#fill-gtheta) — the body of
   `g_theta(uh, r_in, r_out)`.

The names are given, so you can see how many objects are wanted; the right hand
sides are yours. Nothing below a blank runs until you have filled it.

It is a starting point. The questions of the subject ask for more: the crack
opening method, the finite difference of the energy, the curve $G(a)$, and the
comparison with the published expressions.

## Libraries

We import DOLFINx and the helper modules of the course. The mesh and the solver
of this homework are in `utils/`.

In [ ]:
import sys
import time

import matplotlib.pyplot as plt
import numpy as np

import dolfinx.fem as fem
import dolfinx.fem.petsc
import dolfinx.plot as plot
import ufl

from mpi4py import MPI
from petsc4py.PETSc import ScalarType

sys.path.append("../utils")

## The mesh

Lengths are normalised by the hole radius, so $R=1$. The quarter runs from
$x=0$ to $x=W$ and from $y=0$ to $y=L$. The crack lies on $x=0$, from $y=1$ to
$y=1+a$. The mesh is refined near the tip and along the hole.

The function that builds it is in `utils/mesh_dcdc.py`. It gives six physical
groups on the boundary:

| group | where | condition |
|---|---|---|
| `crack` | $x=0$, $1<y<1+a$ | free |
| `ligament` | $x=0$, $y>1+a$ | $u_x=0$ (symmetry) |
| `sym` | $y=0$, $x>1$ | $u_y=0$ (symmetry) |
| `end` | $y=L$ | the compression |
| `side` | $x=W$ | free |
| `hole` | the quarter circle | free |

**The specimen must be long.** `long_enough(a, W)` returns $L=R+a+\max(8R,3W)$.
Below that the end face is felt at the crack tip and the answer is not the DCDC
one. Question 2 is at $W=2R$ for every group; Questions 3 and 4 take the $W/R$
of your group, from the parameter sheet.

The mesh carries three element sizes, and the next section says why:
`lc_tip` at the crack tip, `lc_far` in the near field and along the hole, and
`coarsen * lc_far` far from both.

In [ ]:
import gmsh
from dolfinx.io import gmsh as gmshio

import mesh_dcdc
import solve_dcdc as sd

a = 2.0              # crack length, in units of R
W = 2.0              # half width W/R: 2 in Question 2, that of your group in 3 and 4
nu = 0.2             # Poisson ratio, used again further down
L = sd.long_enough(a, W)

mesh_dcdc.build(a=a, half_width=W, half_length=L, lc_far=0.15, lc_tip=a / 40)
data = gmshio.model_to_mesh(gmsh.model, MPI.COMM_SELF, 0, gdim=2)
msh, facets, cells = data.mesh, data.facet_tags, data.cell_tags
gmsh.finalize()

TAGS = mesh_dcdc.TAGS
print(f"a/R = {a}, W/R = {W}, L/R = {L}, {msh.geometry.x.shape[0]} nodes")

### What the mesh carries

A mesh is not only points and triangles: it carries **tagged element sets**, of
both dimensions, and they are what the forms refer to.

* the **cells**, dimension 2, in one group `domain`. This is what `dx`
  integrates over. A geometry built without that group reaches DOLFINx with no
  cell at all;
* the **facets**, dimension 1, in six groups, one per piece of the boundary.
  Each carries one condition, and `ds(TAGS[name])` is the integral over it.

Reading them off the picture is the first thing to do: a condition put on the
wrong group is the commonest way of solving, carefully, a different problem.

`mesh_dcdc.plot_tags` fills the cells of each cell group, draws the segments of
each facet group over them, and puts in the legend the condition each one holds
and how many elements it has. The right panel is the same thing around the hole.
Note that the crack and the ligament lie on the *same* line $x=0$: what
separates them is the condition, and that is what makes the crack a crack.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 7.2))
mesh_dcdc.plot_tags(msh, facets, cells, ax=axes[0], legend_below=True,
                    title="the quarter and its tagged elements")
mesh_dcdc.plot_tags(msh, facets, cells, ax=axes[1], legend=False,
                    zoom=(-0.2, min(W, 1.8), -0.2, 1.0 + 1.3 * a),
                    title="around the hole and the crack")
fig.tight_layout()

## Why the mesh is graded

A long specimen is mostly empty. At $a=10R$ the strip is some twenty radii
long, and all of it except the neighbourhood of the hole and of the crack tip
carries the same thing: the uniform compression $\sigma$, a field a linear
element represents exactly. Meshing that part at the size the crack tip needs
buys nothing and costs everything, because it is where nearly all the area is.

So the mesh grows outwards, in three steps:

| size | where | why |
|---|---|---|
| `lc_tip`, about $a/40$ | at the crack tip | the singularity |
| `lc_far`, about $R/7$ | near field, and along the hole | the hoop stress that drives the crack |
| `coarsen * lc_far` | far from both | the uniform compression, and nothing else |

`coarsen=1` turns the grading off and gives the uniform mesh. The cell below
runs the same problem for four values and prints what grading costs and what it
saves. Turning it off multiplies the degrees of freedom by four or five, and
moves $G$ in the fourth digit.

**Which knob to turn.** Convergence is studied on `lc_tip` and `lc_far`, never
on `coarsen`: the first two decide the accuracy, the third only decides the
price. Take `coarsen` between 4 and 6 and leave it alone.

In [ ]:
print(f"{'coarsen':>8} {'dofs':>8} {'G-theta':>10} {'seconds':>8}")
for cz in (1.0, 2.0, 4.0, 8.0):
    t0 = time.time()
    trial = sd.solve(a, half_width=W, nu=nu, lc_far=0.15, lc_tip=a / 40,
                     coarsen=cz)
    print(f"{cz:8.0f} {trial['dofs']:8d} {sd.g_theta(trial):10.5f} "
          f"{time.time() - t0:8.2f}")

<a id="fill-bcs"></a>

## Finite element space and boundary conditions

We use linear Lagrange triangles. The module `utils/solve_dcdc.py` uses
quadratic ones, which are more accurate near the tip.

**To fill in.** Two conditions, one on each plane of symmetry, and nothing
else. The code that imposes them is written; what is left is, for each, the
component of the displacement it blocks, 0 for $u_x$ and 1 for $u_y$, and the
value it prescribes:

* on the ligament, the part of $x=0$ ahead of the tip, the displacement normal
  to that plane vanishes;
* on `sym`, the part of $y=0$ outside the hole, the displacement normal to
  *that* plane vanishes.

The crack faces get nothing: they are free, and that is the whole difference
between a crack and a line of symmetry. Together the two conditions remove
every rigid motion, so no node has to be fixed by hand.

In the cell, `fem.locate_dofs_topological(V.sub(i), fdim, facets.find(TAGS[name]))`
finds the degrees of freedom of component `i` on the group `name`, and
`fem.dirichletbc(value, dofs, V.sub(i))` prescribes `value` on them.

In [ ]:
V = fem.functionspace(msh, ("Lagrange", 1, (2,)))
fdim = msh.topology.dim - 1

# ---- TO FILL IN: the two conditions of symmetry
# On each plane of symmetry one displacement component is prescribed. Give
# which one, 0 for u_x and 1 for u_y, and the value it takes, a number.
ligament_component = ...   # on the ligament: x = 0, ahead of the tip
ligament_value = ...
sym_component = ...        # on sym: y = 0, outside the hole
sym_value = ...

ligament_dofs = fem.locate_dofs_topological(
    V.sub(ligament_component), fdim, facets.find(TAGS["ligament"]))
sym_dofs = fem.locate_dofs_topological(
    V.sub(sym_component), fdim, facets.find(TAGS["sym"]))
bcs = [fem.dirichletbc(ScalarType(ligament_value), ligament_dofs,
                       V.sub(ligament_component)),
       fem.dirichletbc(ScalarType(sym_value), sym_dofs, V.sub(sym_component))]

print(f"{len(ligament_dofs)} dofs blocked on the ligament, "
      f"{len(sym_dofs)} on the lower symmetry plane")

<a id="fill-hooke"></a>
<a id="fill-loading"></a>

## The material and the loading

**To fill in.** Two things.

1. The stress `sigma(e)` of the isotropic linear elastic material, from the
   strain `e`. The cell computes the Lamé coefficients `mu` and `lmbda` of
   plane strain; write Hooke's law with them.
2. The traction on the end face $y=L$, where the specimen carries a uniform
   compression of magnitude $\sigma=1$: the two components of the traction
   vector $\underline{T}$ that acts there. It is a compression, and the
   specimen is loaded along its long axis: both the sign and the direction
   matter, and getting either wrong gives a specimen that is pulled, or
   sheared, and a crack that closes.

Nothing constrains $u_x$ on that face: the platens are frictionless. The end
face is the only loaded boundary, and it must stay far from the crack tip,
which is what the length $L$ above is for.

In [ ]:
E = 1.0
mu = E / (2.0 * (1.0 + nu))
lmbda = E * nu / ((1.0 + nu) * (1.0 - 2.0 * nu))   # plane strain
E_eff = E / (1.0 - nu**2)                          # E' of the Irwin relation


def eps(u):
    """The linearised strain of the displacement `u`.

    eps(u) = (grad u + grad u^T) / 2, the symmetric part of the displacement
    gradient: the strain of small deformations. `u` is a vector field and the
    result a 2 x 2 symmetric tensor, both ufl expressions.
    """
    return ufl.sym(ufl.grad(u))


def sigma(e):
    """The stress of the isotropic linear elastic material, from the strain `e`.

    `e` is the in-plane strain, a 2 x 2 symmetric ufl tensor; the result is the
    in-plane stress, a 2 x 2 ufl tensor too. The material enters through the
    Lame coefficients `mu` and `lmbda` above, those of plane strain. In ufl,
    `ufl.tr(e)` is the trace of `e` and `ufl.Identity(2)` the identity.
    """
    # ---- TO FILL IN: Hooke's law: the stress from the strain e, with mu and lmbda
    return ...


# ---- TO FILL IN: the traction on the end face y = L
# T is the force per unit area that the platen exerts on the specimen through
# the face y = L, for a compression of magnitude sigma = 1. Give its two
# components, two numbers: mind the sign.
T_x = ...   # along x, across the specimen
T_y = ...   # along y, the axis of the load
traction = fem.Constant(msh, ScalarType((T_x, T_y)))

dx = ufl.Measure("dx", domain=msh)
ds = ufl.Measure("ds", domain=msh, subdomain_data=facets)


def a_form(u, v):
    """The bilinear form of the elastic problem.

    a(u, v) = int sigma(eps(u)) : eps(v) dx over the meshed quarter: the
    virtual work of the stress of `u` in the strain of `v`. With v = u it is
    twice the elastic energy stored in the quarter.
    """
    return ufl.inner(sigma(eps(u)), eps(v)) * dx


def L_form(v):
    """The linear form of the elastic problem.

    L(v) = int T . v ds over the end face y = L: the virtual work of the
    traction `traction` in the displacement `v`. The end face is the only
    loaded boundary: the hole, the side and the crack faces are free, and the
    two planes of symmetry carry the displacement conditions.
    """
    return ufl.dot(traction, v) * ds(TAGS["end"])

## Solve

The finite element discretisation turns the variational problem
$a(u,v)=L(v)$, for every admissible $v$, into a linear system $K\,U=F$. $K$ is
the stiffness matrix, assembled from `a_form`; $F$ is the load vector,
assembled from `L_form`; $U$ holds the nodal displacements. The Dirichlet
conditions `bcs` are applied to the system during the assembly.

`fem.petsc.LinearProblem` assembles $K$ and $F$ as objects of
[PETSc](https://petsc.org), the linear algebra library on which DOLFINx
relies, and solves the system with a PETSc linear solver, a *KSP* object. The
options passed in `petsc_options` choose that solver:

| option | value | meaning |
|---|---|---|
| `ksp_type` | `preonly` | no Krylov iteration: the preconditioner is applied once |
| `pc_type` | `lu` | the preconditioner is the LU factorisation of $K$ |

Together they give a direct solver: $K$ is factorised once, $K=LU$, and $U$
follows from two triangular solves. The result is exact up to round-off, and
no tolerance is needed. The memory a factorisation takes grows faster than the
number of unknowns; at the few tens of thousands of unknowns of this problem it
is the simplest and fastest choice. Large problems, in three dimensions, use an
iterative solver instead, for instance `"ksp_type": "cg"` with an algebraic
multigrid preconditioner, `"pc_type": "gamg"`.

`petsc_options_prefix` names the options of this solver, so that they do not
mix with those of another one: the projection further down uses `projection_`.

In [ ]:
u, v = ufl.TrialFunction(V), ufl.TestFunction(V)

# K U = F, assembled and solved by PETSc
problem = fem.petsc.LinearProblem(
    a_form(u, v), L_form(v), bcs=bcs, petsc_options_prefix="hw2026_",
    petsc_options={"ksp_type": "preonly",   # no Krylov iteration,
                   "pc_type": "lu"})        # the LU factorisation of K: direct
uh = problem.solve()
uh.name = "displacement"

<a id="fill-energy"></a>

## The potential energy

$\mathcal{P}=\frac12\,a(u,u)-L(u)$, which at equilibrium is $-\frac12 L(u)$.
The forms `a_form` and `L_form` integrate over the mesh, that is over the
**quarter**.

**What a form is.** In UFL, the language in which DOLFINx writes variational
problems, a *form* is an integral written symbolically: an integrand, built from
functions and their derivatives, times a measure, `dx` over the cells or
`ds(TAGS[name])` over a group of facets. Writing it computes nothing. A form is
linear in each of its *arguments*, the `TrialFunction` `u` and the
`TestFunction` `v`, and their number sets what its assembly returns:

| arguments | example | assembled into |
|---|---|---|
| two | `a_form(u, v)` | a matrix, the stiffness $K$ |
| one | `L_form(v)` | a vector, the load $F$ |
| none | a form in which only a `Function`, such as `uh`, appears | a number |

**To fill in.** The function `potential_energy(uh)`: the potential energy of
the quarter as a ufl form, then that of the whole specimen. Two symmetries, so
four quarters, all storing the same thing.

`fem.assemble_scalar(fem.form(form))` computes the integral of a form on the
part of the mesh that the current process holds. A computation run in parallel
with MPI splits the mesh among several processes, and
`msh.comm.allreduce(value, op=MPI.SUM)` adds up the parts and gives the total
to every process. In serial it returns the value unchanged.

Keep the other factor in mind for later, because it does not appear here: the
specimen has *two* crack tips and they grow together, so the energy release
rate of one of them is $G=-\frac12\,\mathrm{d}\Pi/\mathrm{d}a$, not
$-\mathrm{d}\Pi/\mathrm{d}a$. A result out by a factor 2 or 4, with a curve of
the right shape, has met one of these two.

In [ ]:
def potential_energy(uh):
    """The potential energy of the whole specimen, for the displacement `uh`.

    P(u) = 1/2 a(u, u) - L(u), with the bilinear form `a_form` and the linear
    form `L_form` above. Both integrate over the mesh, which holds one quarter
    of the specimen. Returns a number.
    """
    # ---- TO FILL IN: the potential energy of the quarter, as a ufl form
    # a_form(u, v) and L_form(v) are ufl forms; a sum of forms, or a form times
    # a number, is a form too.
    form = ...
    # the integral of the form over the quarter, summed over the MPI processes
    quarter = msh.comm.allreduce(fem.assemble_scalar(fem.form(form)), op=MPI.SUM)
    # ---- TO FILL IN: the potential energy of the whole specimen, from that of the quarter
    return ...


energy = potential_energy(uh)
end_u = msh.comm.allreduce(
    fem.assemble_scalar(fem.form(-uh[1] * ds(TAGS["end"]))), op=MPI.SUM)
shortening = 2.0 * end_u / W         # mean shortening of the whole specimen

print(f"potential energy of the specimen  : {energy: .6e}")
print(f"mean shortening of the end faces  : {shortening: .6e}")

## The deformed shape and the von Mises stress

In [ ]:
import pyvista

pyvista.set_jupyter_backend("static")

sigma_iso = 1.0 / 3 * ufl.tr(sigma(eps(uh))) * ufl.Identity(2)
sigma_dev = sigma(eps(uh)) - sigma_iso
von_mises = ufl.sqrt(3.0 / 2 * ufl.inner(sigma_dev, sigma_dev))

V_vm = fem.functionspace(msh, ("DG", 0))
vm = fem.Function(V_vm)
vm.interpolate(fem.Expression(von_mises, V_vm.element.interpolation_points))

from plots import warp_plot_2d

plotter = warp_plot_2d(uh, cell_field=vm, field_name="von Mises stress",
                       factor=0.2, show_edges=False)
plotter.window_size = [400, 800]
if not pyvista.OFF_SCREEN:
    plotter.show()

<a id="fill-opening"></a>

## Opening and stress along the crack line

On the line $x=0$:

* $u_x$ is half of the crack opening for $1<y<1+a$, and is zero for $y>1+a$,
  where the symmetry condition holds;
* $\sigma_{xx}$ does the opposite: it is zero on the free faces of the crack,
  and it is unbounded at the tip.

The function below projects a stress component on a $P_1$ space, so that it can
be evaluated at points, and returns the values along a line. It is the one place
where the mesh size shows directly: the peak of $\sigma_{xx}$ is not a number to
trust, it only grows when the mesh is refined.

**To fill in.** The function `crack_opening(uh, points)`: the crack opening
$\delta$ at points of the line $x=0$, from the $u_x$ read there. The mesh holds
one half of the crack, and the opening is the distance between its two faces.

The value of $\sigma_{xx}$ at the mouth of the hole, $y=1$, read on the
specimen **without** crack, is the stress concentration $k_t$. The short crack
check of Question 3 takes the hoop stress $s=k_t\,\sigma$.

In [ ]:
from evaluation import evaluate_function


def along_line(field_ufl, points, name="field"):
    "Values of a ufl scalar expression at given points."
    # The displacement is P1, so a stress component is constant over each
    # cell. It is projected on P1 and not interpolated: interpolation would
    # give each node the value of whichever neighbouring cell was written
    # last.
    V_s = fem.functionspace(msh, ("Lagrange", 1))
    p, q = ufl.TrialFunction(V_s), ufl.TestFunction(V_s)
    projection = fem.petsc.LinearProblem(
        ufl.inner(p, q) * dx, ufl.inner(field_ufl, q) * dx,
        petsc_options_prefix="projection_",
        petsc_options={"ksp_type": "preonly", "pc_type": "lu"})
    field = projection.solve()
    field.name = name
    return evaluate_function(field, points)[:, 0]


ys = np.linspace(1.0, min(1.0 + 4.0 * a, L), 501)
points = np.zeros((ys.size, 2))
points[:, 1] = ys

s_xx = along_line(sigma(eps(uh))[0, 0], points, "sigma_xx")


def crack_opening(uh, points):
    """The crack opening at `points`, an (n, 2) array of points of x = 0.

    The mesh holds the half x > 0 of the specimen, and with it one face of the
    crack. The opening is the distance between the two faces of the crack,
    normal to its plane. Returns an array of n numbers.
    """
    u_x = evaluate_function(uh, points)[:, 0]     # u_x at the points
    # ---- TO FILL IN: the opening, from u_x
    # `u_x` is an array of n numbers: the displacement u_x of the crack face
    # the mesh holds.
    return ...


opening = crack_opening(uh, points)

print(f"opening at the mouth      : delta(0,1) = {opening[0]:.4f}")
print(f"largest sigma_xx on x = 0 : {s_xx.max():.3f} "
      f"at y = {ys[s_xx.argmax()]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].plot(ys, opening)
axes[0].set_xlabel("$y/R$")
axes[0].set_ylabel(r"crack opening $\delta$ on $x=0$")

axes[1].plot(ys, s_xx)
axes[1].set_xlabel("$y/R$")
axes[1].set_ylabel(r"$\sigma_{xx}$ on $x=0$")

for ax in axes:
    ax.axvline(1.0 + a, color="k", linestyle="--", linewidth=0.8)
    ax.annotate("crack tip", (1.0 + a, ax.get_ylim()[1]), xytext=(4, -12),
                textcoords="offset points", fontsize=9)

fig.tight_layout()

<a id="fill-gtheta"></a>

## $G$ by the $G$--$\theta$ method

The virtual advance field $\underline\theta$ moves the crack tip along the path
that the crack can take. It is equal to one near the tip, zero far from it, and
it goes down linearly in between. The scalar ramp `th` is built for you; what it
multiplies is not.

With the elastic energy density
$w=\frac12\,\underline{\underline\sigma}:\underline{\underline\varepsilon}$ and
the Eshelby tensor
$\Sigma = w\,\underline{\underline I} - \nabla\underline u^{T}\!\cdot
\underline{\underline\sigma}$,

$$G = -\int_\Omega \Sigma : \nabla\underline\theta \,\mathrm{d}x .$$

**To fill in.** The body of `g_theta`, below the ramp `th`:

1. the vector field $\underline\theta$, from the scalar ramp `th`. It must
   point along the path the crack can take, and that path is not the $x$ axis;
2. the elastic energy density $w$;
3. the Eshelby tensor $\Sigma$;
4. the form $-\Sigma:\nabla\underline\theta\,\mathrm{d}x$;
5. what the function returns. The integral is taken over the mesh, that is over
   the half disc $x>0$, while $\underline\theta$ moves a tip that has material
   on both sides of the crack plane.

The result must not depend on where the ramp is put, as long as it encloses the
tip. The loop below checks this: it is the cheapest test of the computation, and
it is also the fastest way to find out that one of the lines above is
wrong.

In [ ]:
def g_theta(uh, r_in, r_out):
    """G of one tip by the G-theta method, for a ramp between r_in and r_out.

    G = - int Sigma : grad(theta) dx, with
      w     = 1/2 sigma : eps           the elastic energy density,
      Sigma = w I - grad(u)^T . sigma   the Eshelby tensor,
      theta = th t                      the virtual advance of the tip,
    th the scalar ramp built below, 1 within r_in of the tip and 0 beyond
    r_out, and t the unit vector along the path the crack can take. The
    integral runs over the mesh, the half disc x > 0. Returns a number.
    """
    # The ramp must fit between the crack tip and every boundary: the hole
    # BEHIND the tip, at the distance a, the free side x = W beside it, and
    # the loaded face y = L ahead of it. If it reaches one of them, theta is
    # no longer zero there, the integral loses its meaning and returns a
    # value that is too small, without any error message. Use
    # min(a, W, L - 1 - a) as the scale, not a.
    V_s = fem.functionspace(msh, ("Lagrange", 1))
    th = fem.Function(V_s)
    th.interpolate(lambda x: np.clip(
        (r_out - np.sqrt(x[0] ** 2 + (x[1] - 1.0 - a) ** 2)) / (r_out - r_in),
        0.0, 1.0))

    # ---- TO FILL IN: theta, the energy density, the Eshelby tensor and the G-theta form
    # In ufl: ufl.as_vector((p, q)) is the vector (p, q), ufl.grad(f) the
    # gradient of f, ufl.inner(A, B) is A : B, ufl.dot(A, B) is A . B, A.T the
    # transpose of A, ufl.Identity(2) the identity, and sigma(eps(uh)) the
    # stress of uh.
    theta_field = ...
    w = ...
    eshelby = ...
    form = ...
    # the integral of the form over the half disc, summed over the MPI processes
    integral = msh.comm.allreduce(fem.assemble_scalar(fem.form(form)), op=MPI.SUM)
    # ---- TO FILL IN: G of one tip, from the integral over the half disc x > 0
    return ...


span = min(a, W, L - 1.0 - a)       # see the warning in the comment above
print("ramp (r_in, r_out)        G")
for r_in, r_out in [(0.15, 0.45), (0.25, 0.75), (0.40, 0.90)]:
    print(f"  ({r_in:.2f}, {r_out:.2f})          "
          f"{g_theta(uh, r_in * span, r_out * span):.5f}")

## The same computation as a function

Everything above is packed into `utils/solve_dcdc.py`, with quadratic elements
and the three methods of the subject. Use it for the curve $G(a)$: running the
notebook by hand for thirty crack lengths is not the idea. It has none of the
blanks, so it is also how you check what you wrote above: your `g_theta` and its
must agree to the digit the mesh allows.

```python
import solve_dcdc as sd

out = sd.solve(2.0, half_width=2.0, nu=0.2, lc_far=0.15)  # one solution
print(sd.g_theta(out))                                    # domain integral
print(sd.k_from_cod(out))                                 # K_I, crack opening
print(sd.g_finite_difference(2.0, half_width=2.0))        # finite difference
sd.hoop_at_mouth(sd.solve(0.0, half_width=2.0))           # k_t, no crack
```

The module also carries the three published expressions, `he_1995`,
`fett_2005` and `pallares_2009`, written as $K_I=\sigma\sqrt{\pi R}\,F$.
Their second coefficient multiplies $a/R$, not $a/W$.

**What to do next.** Question 2 asks for the three methods at $a=2R$ and for several
mesh sizes. Questions 3 and 4 ask for the curve, from $a/R=0.01$ to 10. Question 5
uses that curve and needs no new computation.

In [ ]:
out = sd.solve(a, half_width=W, nu=nu, lc_far=0.15, lc_tip=a / 40)
print(f"G, G-theta, this notebook : {g_theta(uh, 0.25 * span, 0.75 * span):.5f}")
print(f"G, G-theta, utils/        : {sd.g_theta(out):.5f}")
print(f"G, crack opening          : {sd.k_from_cod(out) ** 2 / out['E_eff']:.5f}")
for name, fit in (("He 1995", sd.he_1995), ("Fett 2005", sd.fett_2005),
                  ("Pallares 2009", sd.pallares_2009)):
    print(f"G, {name:<22s}: {sd.g_handbook(a, W, nu=nu, fit=fit):.5f}")